In [1]:
# !pip install yfinance requests transformers torch xgboost scikit-learn pandas plotly

In [10]:
import yfinance as yf         # API Yahoo Finance 
import requests               # Pour interroger l'API REST HTTP de NewsAPI
import pandas as pd       
import numpy as np           
from datetime import datetime, timedelta # Dates
from transformers import pipeline 
import xgboost as xgb             
from sklearn.ensemble import RandomForestClassifier 
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sqlite3

print("Tous les modules ont été importés avec succès ! L'environnement est prêt.")

Tous les modules ont été importés avec succès ! L'environnement est prêt.


## I. Test Extraction Yahoo Finance

In [3]:
# 1. Définir le symbole boursier (ticker) de l'entreprise (ex: Apple)
ticker = "AAPL"
action = yf.Ticker(ticker)

# 2. Récupérer l'historique des prix sur le dernier mois (1 month)
print(f"Téléchargement des données pour {ticker}...")
df_prix = action.history(period="1mo")

# 3. Filtrer pour garder uniquement les colonnes requises par le projet
colonnes_requises = ["Open", "High", "Low", "Close", "Volume"]
df_prix = df_prix[colonnes_requises]

# 4. Afficher les 5 premières lignes pour vérifier que ça marche
df_prix.head()

Téléchargement des données pour AAPL...


,Open,High,Low,Close,Volume
Date,,,,,
2026-07-22 00:00:00-04:00,327.587466,328.716497,323.061371,325.609192,38755900
2026-07-23 00:00:00-04:00,321.452790,323.021414,319.074836,321.382843,40840800
2026-07-24 00:00:00-04:00,321.512728,334.081875,321.342861,332.733032,47489400
2026-07-27 00:00:00-04:00,334.251737,339.277401,333.732166,336.619690,49604300
2026-07-28 00:00:00-04:00,339.736982,342.594533,335.310806,339.786926,51859000


## II. Test Extraction NewsAPI

In [4]:
# API key for NewsAPI (You need to register on newsapi.org to get a free key)
NEWS_API_KEY = "6bef262549ab4b92a72ab2642be1d7c0" 

def fetch_financial_news(query: str, days_back: int = 7) -> pd.DataFrame:
    """
    Fetches recent news articles related to a specific company or ticker.
    
    Args:
        query (str): The search term (e.g., "Apple" or "AAPL").
        days_back (int): Number of days to look back for news.
        
    Returns:
        pd.DataFrame: A dataframe containing the publication date, title, and summary of the articles.
    """
    # Calculate dates for the API request using datetime and timedelta
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days_back)
    
    # Format dates as strings (YYYY-MM-DD) required by NewsAPI
    from_param = start_date.strftime('%Y-%m-%d')
    to_param = end_date.strftime('%Y-%m-%d')
    
    # NewsAPI endpoint for searching all articles
    url = "https://newsapi.org/v2/everything"
    
    # Parameters for the API request
    params = {
        'q': query,
        'from': from_param,
        'to': to_param,
        'language': 'en',  # English news is mandatory for FinBERT compatibility
        'sortBy': 'relevancy',
        'apiKey': NEWS_API_KEY
    }
    
    # Send the HTTP GET request to the API
    response = requests.get(url, params=params)
    
    # Check if the request was successful (HTTP status code 200)
    if response.status_code == 200:
        data = response.json()
        articles = data.get('articles', [])
        
        # Extract only the relevant fields (Headline and Summary) for our NLP analysis
        extracted_data = []
        for item in articles:
            # Check if title and description exist to avoid NoneType errors
            if item.get('title') and item.get('description'):
                extracted_data.append({
                    'Date': item['publishedAt'][:10], # Keep only the YYYY-MM-DD part
                    'Headline': item['title'],
                    'Summary': item['description']
                })
            
        # Convert the list of dictionaries into a Pandas DataFrame
        df_news = pd.DataFrame(extracted_data)
        return df_news
    else:
        print(f"Error fetching news: {response.status_code} - {response.text}")
        return pd.DataFrame()

# --- Test the news extraction ---
df_apple_news = fetch_financial_news(query="Apple OR AAPL", days_back=7)

# Display the first 5 rows to visualize our text dataset
df_apple_news.head()

,Date,Headline,Summary
0,2026-08-21,"ChatGPT’s Mac App Can Now Control iMessage, Wh...",Even if Apple didn’t watch iMessage like a haw...
1,2026-08-18,Apple squashes EU beef with new App Store rules,Apple is once again overhauling App Store rule...
2,2026-08-21,Walmart is finally adding Apple Pay and Google...,Walmart will soon allow you to pay for your it...
3,2026-08-17,Apple ordered to change app data consent promp...,Apple's changing its rules for data collection...
4,2026-08-18,Unearthed Video Seems to Reveal Apple AirPods ...,Is this the quiet—and presumably accidental—un...


## III. Analyse de Sentiment avec FinBERT

In [5]:
def analyze_financial_sentiment(df: pd.DataFrame, text_column: str = 'Headline') -> pd.DataFrame:
    """
    Applies the FinBERT NLP model to evaluate the sentiment of financial texts.
    
    Args:
        df (pd.DataFrame): The dataframe containing the news articles.
        text_column (str): The column name containing the text to analyze (Headline or Summary).
        
    Returns:
        pd.DataFrame: The original dataframe with two new columns: 'Sentiment_Label' and 'Sentiment_Score'.
    """
    # Check if the dataframe is empty to avoid errors
    if df.empty:
        print("The dataframe is empty. No sentiment analysis performed.")
        return df
        
    print("Loading FinBERT model... (This might take a minute the first time)")
    # Load the pre-trained FinBERT model specialized in finance
    sentiment_analyzer = pipeline("sentiment-analysis", model="ProsusAI/finbert")
    
    # Convert the text column to a list for the pipeline
    texts = df[text_column].tolist()
    
    print(f"Analyzing sentiment for {len(texts)} articles...")
    # Run the model on our list of texts
    results = sentiment_analyzer(texts)
    
    # Extract labels (Positive, Negative, Neutral) and confidence scores
    labels = [res['label'] for res in results]
    scores = [res['score'] for res in results]
    
    # Add the new data to our dataframe
    df_result = df.copy()
    df_result['Sentiment_Label'] = labels
    df_result['Sentiment_Score'] = scores
    
    print("Sentiment analysis completed!")
    return df_result

# --- Run the analysis on the Apple news we extracted previously ---
# Assuming your previous dataframe is named df_apple_news
df_news_with_sentiment = analyze_financial_sentiment(df_apple_news, text_column='Headline')

# Display the results
df_news_with_sentiment.head()

Loading FinBERT model... (This might take a minute the first time)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Analyzing sentiment for 88 articles...
Sentiment analysis completed!


,Date,Headline,Summary,Sentiment_Label,Sentiment_Score
0,2026-08-21,"ChatGPT’s Mac App Can Now Control iMessage, Wh...",Even if Apple didn’t watch iMessage like a haw...,neutral,0.908355
1,2026-08-18,Apple squashes EU beef with new App Store rules,Apple is once again overhauling App Store rule...,neutral,0.757965
2,2026-08-21,Walmart is finally adding Apple Pay and Google...,Walmart will soon allow you to pay for your it...,neutral,0.721018
3,2026-08-17,Apple ordered to change app data consent promp...,Apple's changing its rules for data collection...,neutral,0.715726
4,2026-08-18,Unearthed Video Seems to Reveal Apple AirPods ...,Is this the quiet—and presumably accidental—un...,neutral,0.930801


## IV. Visualisation du Sentiment

In [6]:
def plot_sentiment_distribution(df: pd.DataFrame, ticker: str):
    """
    Creates a beautiful, interactive pie chart of the sentiment distribution using Plotly.
    
    Args:
        df (pd.DataFrame): The dataframe containing the 'Sentiment_Label' column.
        ticker (str): The stock ticker to display in the title.
    """
    if 'Sentiment_Label' not in df.columns:
        print("Error: 'Sentiment_Label' column not found. Run FinBERT first.")
        return
        
    # Count the occurrences of each sentiment
    sentiment_counts = df['Sentiment_Label'].value_counts().reset_index()
    sentiment_counts.columns = ['Sentiment', 'Count']
    
    # Define a custom color palette for finance (Green=Positive, Red=Negative, Grey=Neutral)
    color_map = {
        'positive': '#2ECC71',  # Emerald Green
        'negative': '#E74C3C',  # Alizarin Red
        'neutral': '#95A5A6'    # Concrete Grey
    }
    
    # Create an interactive pie chart
    fig = px.pie(
        sentiment_counts, 
        names='Sentiment', 
        values='Count',
        title=f"<b>News Sentiment Distribution for {ticker} (Last 7 Days)</b>",
        color='Sentiment',
        color_discrete_map=color_map,
        hole=0.4 # Turns the pie chart into an elegant donut chart
    )
    
    # Upgrade the visual layout for a professional dashboard look
    fig.update_layout(
        title_font_size=20,
        font=dict(family="Arial, sans-serif", size=14),
        annotations=[dict(text='Sentiment', x=0.5, y=0.5, font_size=18, showarrow=False)]
    )
    
    # Display the interactive chart
    fig.show()

# --- Display the visual ---
plot_sentiment_distribution(df_news_with_sentiment, ticker="APPL")

## V. Feature Engineering : Indicateurs Techniques

In [7]:
def calculate_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates essential technical indicators: SMA, EMA, Volatility, and RSI.
    
    Args:
        df (pd.DataFrame): Financial dataset containing at least a 'Close' column.
        
    Returns:
        pd.DataFrame: A new dataframe enriched with technical features.
    """
    # Create a copy to avoid modifying the original data
    df_feat = df.copy()
    
    # 1. Moving Averages (20 days window)
    df_feat['SMA_20'] = df_feat['Close'].rolling(window=20).mean()
    df_feat['EMA_20'] = df_feat['Close'].ewm(span=20, adjust=False).mean()
    
    # 2. Daily Returns & Historical Volatility (14 days rolling standard deviation)
    # np.sqrt(252) is used to annualize the volatility (252 trading days in a year)
    df_feat['Daily_Return'] = df_feat['Close'].pct_change()
    df_feat['Volatility_14'] = df_feat['Daily_Return'].rolling(window=14).std() * np.sqrt(252)
    
    # 3. Relative Strength Index (RSI - 14 days)
    # Calculate the daily price differences
    delta = df_feat['Close'].diff()
    
    # Separate the gains and the losses
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    
    # Calculate the Relative Strength (RS) and RSI
    rs = gain / loss
    df_feat['RSI_14'] = 100 - (100 / (1 + rs))
    
    return df_feat

def plot_technical_dashboard(df: pd.DataFrame, ticker: str):
    """
    Generates a professional financial dashboard with Price, MAs, and RSI.
    """
    # Create a subplot grid: 2 rows, 1 column. 
    # The top chart (price) takes 70% of the space, bottom (RSI) takes 30%.
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        row_heights=[0.7, 0.3],
                        vertical_spacing=0.05,
                        subplot_titles=(f"{ticker} - Price & Moving Averages", "Relative Strength Index (RSI)"))
                        
    # Top Chart: Price and Moving Averages
    fig.add_trace(go.Scatter(x=df.index, y=df['Close'], name='Close Price', line=dict(color='white', width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['SMA_20'], name='SMA 20', line=dict(color='#00F5FF', width=1.5)), row=1, col=1) # Cyan
    fig.add_trace(go.Scatter(x=df.index, y=df['EMA_20'], name='EMA 20', line=dict(color='#FF9900', width=1.5, dash='dot')), row=1, col=1) # Orange
    
    # Bottom Chart: RSI
    fig.add_trace(go.Scatter(x=df.index, y=df['RSI_14'], name='RSI 14', line=dict(color='#FF00FF', width=1.5)), row=2, col=1) # Magenta
    
    # Add Overbought (70) and Oversold (30) reference lines for RSI
    fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1, annotation_text="Overbought (70)")
    fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1, annotation_text="Oversold (30)")
    
    # Enhance the visual theme (Dark mode for a professional trading platform look)
    fig.update_layout(
        template='plotly_dark',
        height=700,
        hovermode='x unified', # Shows a single tooltip for all lines when hovering
        margin=dict(l=20, r=20, t=50, b=20)
    )
    
    fig.show()

# --- Execution & Test ---
# We download 6 months of data this time so the 20-day averages have enough data to compute properly
action_test = yf.Ticker("AAPL")
df_prix_6mo = action_test.history(period="6mo")[["Open", "High", "Low", "Close", "Volume"]]

# Calculate indicators
df_features = calculate_technical_indicators(df_prix_6mo)

# Display the beautiful visual
plot_technical_dashboard(df_features.tail(100), ticker="AAPL") # .tail(100) shows the last 100 days for better readability

## VI. Création de la Variable Cible (Target)

In [8]:
def create_target_variable(df: pd.DataFrame, horizon: int = 5) -> pd.DataFrame:
    """
    Creates a binary target variable for Machine Learning classification.
    
    Args:
        df (pd.DataFrame): The financial dataset containing the 'Close' price.
        horizon (int): The number of days in the future to look at (default is 5).
        
    Returns:
        pd.DataFrame: Dataset with 'Future_Close' and 'Target' columns.
                      Drops the last 'horizon' rows as their future is unknown.
    """
    df_target = df.copy()
    
    # 1. Shift the Close price backwards by 'horizon' days to align it with today's row
    df_target['Future_Close'] = df_target['Close'].shift(-horizon)
    
    # 2. Create the binary target: 1 if Future_Close > Current Close, else 0
    # We use .astype(int) to convert boolean (True/False) to 1/0
    df_target['Target'] = (df_target['Future_Close'] > df_target['Close']).astype(int)
    
    # 3. Drop rows with missing future values (the last 5 days of our dataset)
    # We can't train the model on these rows because we don't know the answer yet!
    df_target = df_target.dropna(subset=['Future_Close'])
    
    return df_target

# --- Execution ---
# Using the df_features created in the previous step
df_model_ready = create_target_variable(df_features, horizon=5)

# Display the logic to verify (Current Close vs Future Close -> Target)
df_model_ready[['Close', 'Future_Close', 'Target']].tail()

,Close,Future_Close,Target
Date,,,
2026-08-10 00:00:00-04:00,308.260010,305.589996,0
2026-08-11 00:00:00-04:00,304.910004,310.029999,1
2026-08-12 00:00:00-04:00,302.250000,316.829987,1
2026-08-13 00:00:00-04:00,305.260010,311.299988,1
2026-08-14 00:00:00-04:00,305.929993,309.350006,1


In [9]:
def plot_target_balance(df: pd.DataFrame, ticker: str):
    """
    Plots a beautiful donut chart to visualize the balance between Up and Down days.
    """
    target_counts = df['Target'].value_counts().reset_index()
    # Map 1 and 0 to readable labels
    target_counts['Target'] = target_counts['Target'].map({1: 'Up (1)', 0: 'Down (0)'})
    target_counts.columns = ['Trend', 'Count']
    
    color_map = {'Up (1)': '#2ECC71', 'Down (0)': '#E74C3C'}
    
    fig = px.pie(
        target_counts, 
        names='Trend', 
        values='Count',
        title=f"<b>Target Class Balance (5-Day Horizon) for {ticker}</b>",
        color='Trend',
        color_discrete_map=color_map,
        hole=0.5
    )
    
    fig.update_layout(template='plotly_dark', title_font_size=20)
    fig.show()

# --- Display ---
plot_target_balance(df_model_ready, ticker="AAPL")

## VII.  Création de la Base de Données SQL

In [11]:
def initialize_database(db_name: str = "finance_nlp.db"):
    """
    Connects to an SQLite database (creates it if it doesn't exist) 
    and initializes the tables according to the star schema.
    """
    # 1. Connect to the database (this creates the file in your working directory)
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    print(f"Connected to database: {db_name}")
    
    # 2. Create the Dimension Table (dim_assets)
    # This stores the static info about the companies we track
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_assets (
        asset_id INTEGER PRIMARY KEY AUTOINCREMENT,
        ticker VARCHAR(10) NOT NULL UNIQUE,
        company_name VARCHAR(100),
        sector VARCHAR(50)
    );
    ''')
    print("Table 'dim_assets' checked/created.")
    
    # 3. Create the Fact Table for Market Data (fact_market_data)
    # Notice how we enforce uniqueness with a composite PRIMARY KEY (date, asset_id)
    # This prevents inserting the same price twice for the same day.
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS fact_market_data (
        date DATE,
        asset_id INTEGER,
        open_price DECIMAL(10,2),
        close_price DECIMAL(10,2),
        volume BIGINT,
        FOREIGN KEY (asset_id) REFERENCES dim_assets(asset_id),
        PRIMARY KEY (date, asset_id)
    );
    ''')
    print("Table 'fact_market_data' checked/created.")
    
    # 4. Create the Fact Table for News & Sentiment (fact_news_sentiment)
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS fact_news_sentiment (
        news_id INTEGER PRIMARY KEY AUTOINCREMENT,
        date DATE,
        asset_id INTEGER,
        headline TEXT,
        sentiment_score DECIMAL(5,4),
        sentiment_label VARCHAR(10),
        FOREIGN KEY (asset_id) REFERENCES dim_assets(asset_id)
    );
    ''')
    print("Table 'fact_news_sentiment' checked/created.")
    
    # 5. Commit the changes and close the connection
    conn.commit()
    conn.close()
    print("Database initialization complete.")

# --- Execution ---
initialize_database()

Connected to database: finance_nlp.db
Table 'dim_assets' checked/created.
Table 'fact_market_data' checked/created.
Table 'fact_news_sentiment' checked/created.
Database initialization complete.


## VIII. Insertion des données dans SQL (Load)

In [12]:
def insert_asset_dimension(conn: sqlite3.Connection, ticker: str, company_name: str, sector: str):
    """
    Inserts or ignores a company into the dim_assets table.
    Returns the asset_id for the given ticker.
    """
    cursor = conn.cursor()
    # INSERT OR IGNORE handles the case where the ticker already exists
    cursor.execute('''
        INSERT OR IGNORE INTO dim_assets (ticker, company_name, sector) 
        VALUES (?, ?, ?)
    ''', (ticker, company_name, sector))
    conn.commit()
    
    # Retrieve and return the asset_id
    cursor.execute('SELECT asset_id FROM dim_assets WHERE ticker = ?', (ticker,))
    result = cursor.fetchone()
    return result[0] if result else None

def insert_market_data(conn: sqlite3.Connection, df_market: pd.DataFrame, asset_id: int):
    """
    Inserts daily OHLCV data into fact_market_data.
    """
    if df_market.empty:
        return
        
    cursor = conn.cursor()
    # Convert dataframe to a list of tuples for fast bulk insertion
    records_to_insert = []
    
    for date, row in df_market.iterrows():
        # Date must be a string (YYYY-MM-DD)
        date_str = date.strftime('%Y-%m-%d')
        records_to_insert.append((
            date_str, 
            asset_id, 
            float(row['Open']), 
            float(row['Close']), 
            int(row['Volume'])
        ))
        
    # INSERT OR IGNORE prevents errors if we try to insert the same day twice
    cursor.executemany('''
        INSERT OR IGNORE INTO fact_market_data (date, asset_id, open_price, close_price, volume)
        VALUES (?, ?, ?, ?, ?)
    ''', records_to_insert)
    
    conn.commit()
    print(f"Inserted {cursor.rowcount} new price records for asset_id {asset_id}.")

# --- Execution ---
# Connect to our database
db_connection = sqlite3.connect("finance_nlp.db")

# 1. Insert Apple into the dimension table and get its ID
apple_id = insert_asset_dimension(
    conn=db_connection, 
    ticker="AAPL", 
    company_name="Apple Inc.", 
    sector="Technology"
)
print(f"Apple is registered in dim_assets with ID: {apple_id}")

# 2. Insert the historical prices we downloaded earlier (df_prix_6mo)
insert_market_data(conn=db_connection, df_market=df_prix_6mo, asset_id=apple_id)

# Always close the connection when done
db_connection.close()

Apple is registered in dim_assets with ID: 1
Inserted 126 new price records for asset_id 1.
